# Task 0: Getting Started

In [1]:
import pandas as pd 
import json
import random 
import time
import hashlib
import datetime
import os
from confluent_kafka import Producer

# Task 1: Build a catalog of products

In [2]:
df = pd.read_csv("usercode/TV_DATASET_USA.csv")

In [3]:
df['BRAND'].value_counts()

SAMSUNG        182
LG             122
SONY            88
TCL             37
INSIGNIA™       34
VIZIO           24
HISENSE         20
ROKU            19
TOSHIBA         13
AMAZON           9
PEERLESS-AV      4
PIONEER          3
FURRION          1
SHARP            1
Name: BRAND, dtype: int64

In [4]:
df['PRICING'] = df['PRICING'].apply(lambda x : float(x.replace('$','').replace(',','')))

# Task 2: Set Up Variables and Functions

In [5]:
cities = ['New York','Los Ángeles','Chicago','Houston','Philadelphia']

payment_modes = ['Credit_card','Stripe','Paypal','Apple_Pay','Google_Pay','Samsung_Pay']

payment_store = ['Cash','Credit_card']

sources = ['Facebook','Instagram','Organic','Twitter','Influencer_1','Influencer_2','Influencer_3','Influencer_4']

purchase_statuses = ['COMPLETED','FAILED_CHECKOUT','FAILED_API_RESPONSE','INSUFICCIENT_FUNDS','COMPLETED','COMPLETED','COMPLETED','COMPLETED','COMPLETED','COMPLETED','FAILED_API_RESPONSE','INSUFICCIENT_FUNDS','USER_ERROR','FRAUD','COMPLETED','COMPLETED','COMPLETED']

commission = [0.2,0.25,0.3,0.27,0.35,0.4,0.37,0.15,0.1]

NY_coords = [(40.76046814557239, -73.97764793953105),(40.76921169592604, -73.98326984936075),(40.762515994719834, -73.98095242088134)]
LA_coords = [(34.07210945806006, -118.35747350374957),(34.071754649810025, -118.37593530089991)]
CHI_coords = [(41.89819876058171, -87.62280110486684),(41.89182575694393, -87.6249468719774),(41.88375296758592, -87.62814652743663)]
HOU_coords = [(29.742233338438325, -95.44654054545151),(29.743148850926094, -95.45312636612748),(29.739981565214627, -95.46428435510245)]
PHI_coords = [(40.089499621312456, -75.39015007888118),(40.085310197975055, -75.40444450974655),(40.09069475292698, -75.3815277170056)]

def get_pay_method(sources,purchase_statuses,payment_modes,payment_store):
    if sources == 'Organic':
        payment = random.choice(payment_store)
        status = 'COMPLETED'
        order_type = 'STORE'
    elif sources != 'Organic':
        payment = random.choice(payment_modes)
        status = random.choice(purchase_statuses)
        order_type = 'ONLINE'
    return payment,status,order_type

def get_coords(city):
    if city == 'New York':
        coords = random.choice(NY_coords)
    elif city == 'Los Ángeles':
        coords = random.choice(LA_coords)
    elif city == 'Chicago':
        coords = random.choice(CHI_coords)
    elif city == 'Houston':
        coords = random.choice(HOU_coords)
    elif city == 'Philadelphia':
        coords = random.choice(PHI_coords)
    return coords

# Task 3: Generate Records

In [6]:
delivered_records = 0

x = 1

data_purchase = []

while(x < 10):

    date = pd.to_datetime('today').strftime("%Y-%m-%d %H:%M:%S")
    product = df['PRODUCT_NAME'][random.randint(0,556)]
    pricing =  df[df['PRODUCT_NAME']==product]['PRICING'].values[0]
    commission_temp =  random.choice(commission)
    brand = df[df['PRODUCT_NAME']==product]['BRAND'].values[0]
    screen = df[df['PRODUCT_NAME']==product]['SCREEN_SIZE'].values[0]
    display = df[df['PRODUCT_NAME']==product]['DISPLAY_TYPE'].values[0]
    resolution = df[df['PRODUCT_NAME']==product]['DISPLAY_TYPE'].values[0]
    source_temp = random.choice(sources)
    pay = get_pay_method(source_temp,purchase_statuses,payment_modes,payment_store)
    city = random.choice(cities)

    purchase = {'purchase_ID': str(hashlib.sha256(f"{x} {product} {pricing} {commission_temp} {date} {source_temp} {pay[1]}".encode('utf-8')).hexdigest())[:10],
            'Product_name' : product,
            'Pricing':str(pricing),
            'Commission':str(commission_temp),
            'Revenue' : str(round(pricing*commission_temp,2)),
            'Payment_Mehtod':pay[0],
            'Status' : pay[1],
            'Order_Type' : pay[2],
            'City':city,
            'Location': str(get_coords(city)),
            'Latitud' : str(get_coords(city)[0]) ,
            'Longitud' :  str(get_coords(city)[1]),
            'Source':source_temp,
            'Brand' : brand,
            'Screen' : screen,
            'Display' :  display ,
            'Resolution' : resolution,
            'Created_at': date}

    data_purchase.append(pd.DataFrame(purchase,index =[x]))
    delivered_records += 1
    x += 1
    print(purchase)
    time.sleep(random.choice([1,1.5]))

print('\n')
print("{} messages were produced".format(delivered_records))
print('\n')

{'purchase_ID': 'd6eebf4d24', 'Product_name': 'LG - 65" Class G2 Series OLED evo 4K UHD Smart webOS TV with Gallery Design', 'Pricing': '1999.99', 'Commission': '0.35', 'Revenue': '700.0', 'Payment_Mehtod': 'Apple_Pay', 'Status': 'COMPLETED', 'Order_Type': 'ONLINE', 'City': 'Philadelphia', 'Location': '(40.089499621312456, -75.39015007888118)', 'Latitud': '40.089499621312456', 'Longitud': '-75.40444450974655', 'Source': 'Twitter', 'Brand': 'LG', 'Screen': 65, 'Display': 'OLED', 'Resolution': 'OLED', 'Created_at': '2026-04-03 00:20:33'}
{'purchase_ID': 'eddb28c9d4', 'Product_name': 'Package - Samsung - 65" Class The Frame QLED 4k Smart Tizen TV - Charcoal Black and Q-Series  9.1.4ch  Wireless True Dolby Atmos Soundbar +  Rear Speakers w/ Q-Symphony - Titan Black', 'Pricing': '2599.98', 'Commission': '0.27', 'Revenue': '701.99', 'Payment_Mehtod': 'Stripe', 'Status': 'COMPLETED', 'Order_Type': 'ONLINE', 'City': 'Philadelphia', 'Location': '(40.089499621312456, -75.39015007888118)', 'Latit

# Task 5: Create a Kafka Producer Instance

In [8]:
from kafka_auth import conf

# Create a Kafka Producer instance
producer = Producer(conf)

# Task 6 : Send Records to Kafka Topic

In [9]:
delivered_records = 0
topic = 'ecommerce-topic-1' 
x = 1

data_purchase = []

while True:

    date = pd.to_datetime('today').strftime("%Y-%m-%d %H:%M:%S")
    product = df['PRODUCT_NAME'][random.randint(0,556)]
    pricing =  df[df['PRODUCT_NAME']==product]['PRICING'].values[0]
    commission_temp =  random.choice(commission)
    brand = df[df['PRODUCT_NAME']==product]['BRAND'].values[0]
    screen = df[df['PRODUCT_NAME']==product]['SCREEN_SIZE'].values[0]
    display = df[df['PRODUCT_NAME']==product]['DISPLAY_TYPE'].values[0]
    resolution = df[df['PRODUCT_NAME']==product]['DISPLAY_TYPE'].values[0]
    source_temp = random.choice(sources)
    pay = get_pay_method(source_temp,purchase_statuses,payment_modes,payment_store)
    city = random.choice(cities)

    purchase = {'purchase_ID': str(hashlib.sha256(f"{x} {product} {pricing} {commission_temp} {date} {source_temp} {pay[1]}".encode('utf-8')).hexdigest())[:10],
            'Product_name' : product,
            'Pricing':str(int(pricing)),
            'Commission':str(commission_temp),
            'Revenue' : str(round(pricing*commission_temp,2)),
            'Payment_Mehtod':pay[0],
            'Status' : pay[1],
            'Order_Type' : pay[2],
            'City':city,
            'Location': str(get_coords(city)),
            'Latitud' : str(get_coords(city)[0]) ,
            'Longitud' :  str(get_coords(city)[1]),
            'Source':source_temp,
            'Brand' : brand,
            'Screen' : str(screen),
            'Display' :  display ,
            'Resolution' : str(resolution),
            'Created_at': date}

    data_purchase.append(pd.DataFrame(purchase,index =[x]))
    
    record_key = "Purchase_simulator"
    record_value =  json.dumps(purchase).encode('utf-8')
    print("Producing record: {}\t{}".format(record_key, record_value))
    producer.produce(topic, key=record_key, value=record_value)

    producer.poll(0)
    delivered_records += 1
    x += 1
    
    time.sleep(random.choice([1,1.5]))
    producer.flush()
    

Producing record: Purchase_simulator	b'{"purchase_ID": "9c1d3b6730", "Product_name": "LG - 55\\" Class UQ75 Series LED 4K UHD Smart webOS TV", "Pricing": "379", "Commission": "0.2", "Revenue": "76.0", "Payment_Mehtod": "Credit_card", "Status": "FAILED_API_RESPONSE", "Order_Type": "ONLINE", "City": "Houston", "Location": "(29.739981565214627, -95.46428435510245)", "Latitud": "29.739981565214627", "Longitud": "-95.45312636612748", "Source": "Twitter", "Brand": "LG", "Screen": "55", "Display": "LED", "Resolution": "LED", "Created_at": "2026-04-03 00:40:56"}'
Producing record: Purchase_simulator	b'{"purchase_ID": "dfe8c2253e", "Product_name": "Insignia\\u2122 - 43\\" Class N10 Series LED Full HD TV", "Pricing": "129", "Commission": "0.2", "Revenue": "26.0", "Payment_Mehtod": "Samsung_Pay", "Status": "INSUFICCIENT_FUNDS", "Order_Type": "ONLINE", "City": "Houston", "Location": "(29.739981565214627, -95.46428435510245)", "Latitud": "29.743148850926094", "Longitud": "-95.46428435510245", "Sour